# SoundScape AI — Nature Generator

**Natural, calming, controllable soundscapes · End-to-end generative audio pipeline**

This notebook uses a **pretrained text-conditioned latent diffusion model (AudioLDM 2)** for audio generation. It also audits real recordings from ESC-50, compares original and generated audio, applies gentle mastering, creates seamless longer scenes, and exports a platform-ready WAV library with a JSON manifest.

**Sound classes:** rain, sea waves, wind, flowing water, distant thunder. The default prompts favor quiet, continuous ambience. No city noise or abrupt effects.

**Runtime:** Google Colab with a GPU (recommended). First execution downloads pretrained weights (several GB) and the ESC-50 dataset. The model requires an internet connection and sufficient RAM/VRAM. The CPU fallback can run but is very slow. The notebook reports real failures rather than substituting ungenerated audio.

**Licensing:** ESC-50 is CC BY-NC; check original Freesound attribution and each clip's license before commercial reuse. AudioLDM 2 weights have their own model license; check the model card before deployment.

## Environment setup — Google Colab

1. Select **Runtime → Change runtime type → T4 GPU** (or stronger).
2. Run the installation cell below before importing the model.
3. If Colab asks to restart after installation, restart once and run from the top. **Do not repeatedly upgrade packages.**
4. First run downloads ESC-50 and approximately 4.5 GB of model files.

The versions below reproduce the library versions used in the successful Colab session (Python 3.13.16, Tesla T4). Installation from a fresh runtime has not yet been independently verified. No fine-tuning is performed in this notebook.

In [ ]:
# Versions observed in the successful Google Colab GPU session.
# Do not replace these pins with unbounded `-U` upgrades.
%pip install -q "diffusers==0.42.1" "transformers==5.22.1" "huggingface-hub==1.33.0" "tokenizers==0.23.3" "librosa==0.11.0" "accelerate==1.15.0" "soundfile" "sentencepiece" "safetensors" "jedi>=0.16"

import os, json, gc, random, subprocess, warnings, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import librosa, soundfile as sf
from scipy.signal import resample_poly
import torch
from IPython.display import Audio, display

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT=Path('soundscape_export'); OUTPUT.mkdir(exist_ok=True)
SR=16000
MODEL_ID='cvssp/audioldm2'
MODEL_REVISION='c8e7e189d324425c05c4c2f81214041ef4107983'
print('Device:',DEVICE,'| PyTorch:',torch.__version__)
if DEVICE=='cuda': print('GPU:',torch.cuda.get_device_name(0))

## Method and reproducibility
AudioLDM 2 combines text encoders and a language model for conditioning with a latent diffusion U-Net, a variational autoencoder and a vocoder. Denoising is pretrained inference; no weights are updated here. CVAEs trained on 200 clips and Griffin-Lim are weak baselines for realistic audio. AudioLDM is another latent-diffusion baseline; MusicGen is geared toward music. Their relative quality on these categories has not been measured in this project. The checkpoint is pinned to a Hub revision and is licensed [CC BY-NC-SA 4.0](https://huggingface.co/cvssp/audioldm2); it is not a commercial-ready choice. ESC-50 is [CC BY-NC](https://github.com/karolpiczak/ESC-50) except its identified ESC-10 subset. FSD50K is a larger candidate but requires per-clip license and source review; it is not downloaded or verified here.

Fine-tuning is an experiment, not an established improvement: 200 five-second class-labeled recordings lack detailed captions and are too small to justify a production adaptation. A meaningful experiment requires licensed, source-separated audio and captions, GPU memory and time measurements, held-out sources, a frozen pretrained baseline, and blinded listening. No training, checkpoint, fine-tuning result, or performance claim is reported. Record GPU memory and runtime below when running inference. Use a Colab GPU runtime and run cells in order; initial downloads require internet and substantial disk space. The model and dependencies are loaded at runtime.

## 2 · Data inspection: ESC-50

The reference dataset provides **40 five-second clips per class**, which is insufficient to train a high-fidelity sound generator from scratch. We therefore use it for transparent audio inspection and comparison, while **generation comes from the pretrained diffusion model**. These are distinct roles.

In [ ]:
CLASSES=['rain','sea_waves','wind','pouring_water','thunderstorm']
ROOT=Path('ESC-50')
if not (ROOT/'meta/esc50.csv').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/karolpiczak/ESC-50.git'],check=True)
meta=pd.read_csv(ROOT/'meta/esc50.csv')
meta=meta[meta.category.isin(CLASSES)].copy().reset_index(drop=True)
meta['path']=meta.filename.map(lambda x:str(ROOT/'audio'/x))
assert len(meta)==200 and meta.path.map(os.path.isfile).all()
display(meta.groupby(['category','fold']).size().unstack(fill_value=0))
print('Original recordings:',len(meta))

In [ ]:
# Folds 1-3 train, 4 validation, 5 test; do not split a source across folds.
meta['split'] = meta.fold.map({1:'train', 2:'train', 3:'train', 4:'validation', 5:'test'})
assert meta.groupby('src_file')['split'].nunique().max() == 1
import hashlib
meta['sha256'] = meta.path.map(lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest())
duplicates = meta[meta.duplicated('sha256', keep=False)]
cross_split = duplicates.groupby('sha256')['split'].nunique()
print('Exact duplicate files:', len(duplicates), '| crossing splits:', int((cross_split > 1).sum()))
assert not (cross_split > 1).any(), 'Exact duplicate across splits'
display(pd.crosstab(meta.category, meta.split))
print('This audit does not rule out near duplicates or pretrained-model contamination.')

## 3 · Audio audit and listening

Duration and silence are measured independently. Quiet sections may be valid acoustic content, particularly in water or distant-thunder recordings. No clips are removed automatically.

In [ ]:
def read_audio(path, sr=SR):
    y,_=librosa.load(str(path),sr=sr,mono=True)
    return np.nan_to_num(y).astype(np.float32)

def audio_metrics(y,sr=SR):
    y=np.asarray(y,dtype=np.float32)
    if len(y)==0:return dict(duration_s=0,rms=0,peak=0,quiet_ratio=1,clipping_ratio=0)
    rms_frames=librosa.feature.rms(y=y,frame_length=1024,hop_length=256)[0]
    return dict(duration_s=round(len(y)/sr,3),rms=float(np.sqrt(np.mean(y*y))),
        peak=float(np.max(np.abs(y))),rms_dbfs=float(20*np.log10(max(float(np.sqrt(np.mean(y*y))),1e-12))),quiet_ratio=float(np.mean(rms_frames<0.001)),
        clipping_ratio=float(np.mean(np.abs(y)>=0.999)))

rows=[]
for row in meta.itertuples():
    rows.append(dict(category=row.category,filename=row.filename,fold=row.fold,
                     **audio_metrics(read_audio(row.path))))
audit=pd.DataFrame(rows)
display(audit.groupby('category')[['rms','quiet_ratio','clipping_ratio']].agg(['mean','max']).round(3))
display(audit.sort_values('quiet_ratio',ascending=False).head(10))
audit.to_csv(OUTPUT/'esc50_audio_audit.csv',index=False)
fig,axes=plt.subplots(len(CLASSES),2,figsize=(12,11))
for i,name in enumerate(CLASSES):
    path=meta.loc[meta.category.eq(name),'path'].iloc[0]; y=read_audio(path)
    axes[i,0].plot(np.arange(len(y))/SR,y,lw=.45);axes[i,0].set_title(name+' — waveform')
    spec=librosa.amplitude_to_db(np.abs(librosa.stft(y)),ref=np.max)
    axes[i,1].imshow(spec,aspect='auto',origin='lower',cmap='magma',vmin=-80,vmax=0)
    axes[i,1].set_title(name+' — spectrogram')
plt.tight_layout();plt.show()
for name in CLASSES:
    path=meta.loc[meta.category.eq(name),'path'].iloc[0]
    print('Reference:',name);display(Audio(filename=path))

## 4 · Generation design

The diffusion model generates waveform audio conditioned on text. Each class has a comfort-oriented prompt and a negative prompt discouraging harsh, distracting events. Prompts are creative guidance, **not a guarantee** of comfort or class accuracy.

In [ ]:
PROMPTS={
 'rain':'Gentle steady rain falling outside a cozy window, soft natural rain texture, relaxing continuous ambience, no music, high quality field recording',
 'sea_waves':'Small gentle ocean waves washing softly onto a quiet sandy beach, natural slow rhythmic surf, peaceful continuous ambience, no music, high quality field recording',
 'wind':'Soft warm breeze moving quietly through tree leaves, gentle rustling foliage, peaceful natural ambience, no music, high quality field recording',
 'pouring_water':'Calm continuous stream of fresh water flowing over small smooth stones, gentle babbling brook, peaceful nature ambience, no music, high quality field recording',
 'thunderstorm':'Gentle distant thunder rumbling far away beneath steady soft rain, cozy safe indoor perspective, quiet calming weather ambience, no music, high quality field recording'
}
NEGATIVE='music, speech, voice, singing, siren, alarm, explosion, gunshot, distortion, harsh noise, loud crack, abrupt impact, clipping'
assert set(PROMPTS)==set(CLASSES)
for k,v in PROMPTS.items():print(k,':',v)

## 5 · Load pretrained AudioLDM 2

AudioLDM 2 is a genuine generative deep-learning model; unlike the earlier small CVAE baseline, it has already learned from a much larger audio corpus. Its model checkpoint is downloaded at runtime. This cell must complete successfully before proceeding.

The GPT-2 compatibility correction is integrated into the loading cell. No separate diagnostic or replacement cells are required.

In [ ]:
from diffusers import AudioLDM2Pipeline
DTYPE=torch.float16 if DEVICE=='cuda' else torch.float32
pipe=AudioLDM2Pipeline.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=DTYPE)
pipe=pipe.to(DEVICE)
if DEVICE=='cuda':
    try:pipe.enable_attention_slicing()
    except Exception:pass
MODEL_SR=int(pipe.vae.config.get('sampling_rate',16000)) if hasattr(pipe.vae.config,'get') else 16000
# The AudioLDM2 pipeline exposes the actual output sample rate via the vocoder configuration.
MODEL_SR=int(getattr(pipe.vocoder.config,'sampling_rate',MODEL_SR))
print('Loaded',MODEL_ID,'| Model output sample rate:',MODEL_SR)

# Compatibility fix verified in the successful Colab run:
# AudioLDM2Pipeline expects a GPT2LMHeadModel with generation methods,
# while the pinned checkpoint can load its language model as GPT2Model.
from transformers import GPT2LMHeadModel
if type(pipe.language_model).__name__ == 'GPT2Model':
    old_model = pipe.language_model
    new_model = GPT2LMHeadModel(old_model.config)
    new_model.transformer.load_state_dict(old_model.state_dict(), strict=True)
    new_model = new_model.to(
        device=next(old_model.parameters()).device,
        dtype=next(old_model.parameters()).dtype
    )
    new_model.eval()
    pipe.register_modules(language_model=new_model)
    del old_model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
assert hasattr(pipe.language_model, '_update_model_kwargs_for_generation'), (
    'Incompatible Transformers GPT-2 generation interface'
)
print('Language model:', type(pipe.language_model).__name__)


In [ ]:
print('CUDA available:', torch.cuda.is_available())
if DEVICE == 'cuda':
    print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
    torch.cuda.reset_peak_memory_stats()
inference_started = time.perf_counter()

## 6 · Generate candidate recordings

For each class, generate several independent candidates, audit them, and keep the best *technical* candidate. This is not an automatic comfort classifier: listening remains necessary. The selection discourages silence, clipping and excessive dynamics; it cannot measure perceptual realism.

In [ ]:
CANDIDATES_PER_CLASS=2
DURATION_SECONDS=10
INFERENCE_STEPS=50
GUIDANCE_SCALE=3.5

def make_candidate(category,seed):
    generator=torch.Generator(device=DEVICE).manual_seed(seed)
    result=pipe(prompt=PROMPTS[category],negative_prompt=NEGATIVE,
        audio_length_in_s=DURATION_SECONDS,num_inference_steps=INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,generator=generator)
    y=np.asarray(result.audios[0],dtype=np.float32).reshape(-1)
    if MODEL_SR!=SR:y=librosa.resample(y,orig_sr=MODEL_SR,target_sr=SR)
    if not np.isfinite(y).all(): raise ValueError('Model returned non-finite audio')
    return y.astype(np.float32)

def quality_score(metrics):
    # A technical screening score, not a perceptual or semantic quality score.
    rms=metrics['rms']
    return float(-abs(np.log10(max(rms,1e-6))-np.log10(0.08))
        -2*metrics['quiet_ratio']-8*metrics['clipping_ratio'])

candidate_rows=[]
for index,category in enumerate(CLASSES):
    for j in range(CANDIDATES_PER_CLASS):
        seed=SEED+index*100+j
        y=make_candidate(category,seed)
        path=OUTPUT/f'{category}_candidate_{j+1}.wav'
        sf.write(path,y,SR,subtype='PCM_16')
        stats=audio_metrics(y)
        candidate_rows.append(dict(category=category,seed=seed,file=str(path),
                                   score=quality_score(stats),**stats))
        print(category,'candidate',j+1,'RMS',round(stats['rms'],4),
              'quiet',round(stats['quiet_ratio'],3))
        if DEVICE=='cuda':torch.cuda.empty_cache()
candidates=pd.DataFrame(candidate_rows)
candidates.to_csv(OUTPUT/'candidate_quality.csv',index=False)
display(candidates.sort_values(['category','score'],ascending=[True,False]))

In [ ]:
print('Generation elapsed seconds:', round(time.perf_counter() - inference_started, 1))
if DEVICE == 'cuda':
    print('Peak allocated GPU GiB:', round(torch.cuda.max_memory_allocated() / 2**30, 2))

## 7 · Compare candidates by ear and by spectrogram

Every class shows its candidates with a player, waveform, and spectral view. Listen for metallic artifacts, repetitive hiss, sudden transients and whether the intended sound is recognizable.

In [ ]:
for category in CLASSES:
    print('\n',category.upper())
    subset=candidates[candidates.category.eq(category)].sort_values('score',ascending=False)
    fig,axes=plt.subplots(1,2,figsize=(11,2.8))
    for _,r in subset.iterrows():
        y=read_audio(r.file)
        print(Path(r.file).name,'score',round(r.score,3));display(Audio(y,rate=SR,normalize=False))
    best=read_audio(subset.iloc[0].file)
    axes[0].plot(np.arange(len(best))/SR,best,lw=.35);axes[0].set_title(category+' waveform')
    mel=librosa.feature.melspectrogram(y=best,sr=SR,n_mels=64)
    axes[1].imshow(librosa.power_to_db(mel,ref=np.max),origin='lower',aspect='auto',vmin=-80,vmax=0)
    axes[1].set_title('Log-mel spectrogram');plt.tight_layout();plt.show()

### Listening assessment
Compare each candidate with its reference recording above. Assess category recognition, naturalness, relaxation, artifacts, and repeated transients on a 1-5 scale. Record reviewer and notes outside the notebook if needed. Technical scoring only screens gross defects; it does not establish semantic or perceptual quality. Set a category's seed below after listening. Unspecified categories retain the technical fallback and remain unreviewed.

In [ ]:
REVIEWED_SEEDS = {}  # Example: {'rain': 42}

## 8 · Gentle mastering and seamless ambient looping

The platform needs longer layers. We assemble a longer scene from each generated recording using overlapping segments and equal-power crossfades, then apply a conservative limiter and fades. **This does not generate new information beyond the diffusion output**, but makes the existing generation easier to use as a background layer.

In [ ]:
def soft_master(y,target_rms=0.09,peak_limit=0.88):
    y=np.nan_to_num(np.asarray(y,dtype=np.float32))
    y=y-np.mean(y)
    rms=float(np.sqrt(np.mean(y*y)))
    if rms>1e-6:y=y*min(3.0,target_rms/rms)
    peak=float(np.max(np.abs(y))) if len(y) else 0
    if peak>peak_limit:y=y*(peak_limit/peak)
    return y.astype(np.float32)

def loop_crossfade(y,seconds=60,fade_seconds=1.0,sr=SR):
    y=np.asarray(y,dtype=np.float32)
    target=int(seconds*sr)
    if len(y)<int(2*fade_seconds*sr)+2:raise ValueError('Audio is too short for crossfade')
    overlap=int(fade_seconds*sr)
    # Create a smooth seamless single cycle by blending the original end into its start.
    # Preserve the main body while making the join continuous.
    head=y[:overlap].copy();tail=y[-overlap:].copy()
    theta=np.linspace(0,np.pi/2,overlap,endpoint=True,dtype=np.float32)
    join=tail*np.cos(theta)**2+head*np.sin(theta)**2
    cycle=np.concatenate([join,y[overlap:-overlap]])
    if len(cycle)==0:raise ValueError('Empty loop cycle')
    repeated=np.tile(cycle,int(np.ceil(target/len(cycle)))+1)[:target]
    edge=min(int(.25*sr),len(repeated)//4)
    if edge:
        repeated[:edge]*=np.linspace(0,1,edge)
        repeated[-edge:]*=np.linspace(1,0,edge)
    return repeated.astype(np.float32)

SCENE_SECONDS=60
selected=[]
for category in CLASSES:
    subset=candidates[candidates.category.eq(category)].sort_values('score',ascending=False)
    override=REVIEWED_SEEDS.get(category)
    best=subset.iloc[0] if override is None else subset[subset.seed.eq(override)].iloc[0]
    y=soft_master(read_audio(best.file))
    y=loop_crossfade(y,seconds=SCENE_SECONDS)
    path=OUTPUT/f'{category}_ambient_60s.wav'
    sf.write(path,y,SR,subtype='PCM_16')
    selected.append(dict(category=category,prompt=PROMPTS[category],seed=int(best.seed),
                         wav=path.name,source_candidate=Path(best.file).name,
                         **audio_metrics(y)))
manifest=pd.DataFrame(selected)
display(manifest)
print('Created',len(manifest),'ambient layers')

### Listen to the final 60-second ambient layers

These are generated 10-second recordings extended by audio mastering and looping; they are **not** independently generated 60-second recordings.

In [ ]:
for category in CLASSES:
    final_path = OUTPUT / f'{category}_ambient_60s.wav'
    print('Final ambient layer:', category)
    display(Audio(filename=str(final_path), normalize=False))

## 9 · Platform-ready API and three-layer mixer

The exported files can be loaded by a Gradio, FastAPI, or other application without retraining or loading the diffusion model. Gains are relative linear amplitudes, **not guaranteed perceived loudness percentages**. A final peak limiter prevents clipping.

In [ ]:
def load_layer(category):
    if category not in CLASSES:raise ValueError('Unknown category: '+str(category))
    path=OUTPUT/f'{category}_ambient_60s.wav'
    y,sr=sf.read(path,dtype='float32',always_2d=False)
    if sr!=SR:raise ValueError('Unexpected sample rate')
    return y

def mix_soundscape(levels,seconds=60):
    # Example: {'rain':20,'sea_waves':50,'wind':10}
    if any(not np.isfinite(float(v)) or float(v)<0 or float(v)>100 for v in levels.values()):
        raise ValueError('Levels must be finite and 0..100')
    active={k:float(v) for k,v in levels.items() if float(v)>0}
    if not active:raise ValueError('At least one layer must be audible')
    if any(k not in CLASSES for k in active):raise ValueError('Unknown category')
    if any(v<0 or v>100 for v in active.values()):raise ValueError('Levels must be 0..100')
    n=int(seconds*SR)
    result=np.zeros(n,dtype=np.float32)
    for category,volume in active.items():
        y=load_layer(category)
        if len(y)<n:y=np.tile(y,int(np.ceil(n/len(y))))
        result+=y[:n]*(volume/100.0)
    peak=float(np.max(np.abs(result)))
    if peak>0.9: result *= 0.9/peak
    return result

DEMO_LEVELS={'rain':60,'sea_waves':30,'wind':10}
mix=mix_soundscape(DEMO_LEVELS,seconds=60)
sf.write(OUTPUT/'demo_mix_60s.wav',mix,SR,subtype='PCM_16')
print('Mixed demo:',DEMO_LEVELS);display(Audio(mix,rate=SR,normalize=False))

## Interface and acceptance
Exports are mono PCM16 WAV at 16 kHz. Manifest paths are relative to the manifest directory. The generated 60-second files are repeated crossfaded short clips, not novel long-form generation. Consumers should apply independent linear gains and leave headroom in their mixer. Review the repeat boundary and transitions by listening. RMS dBFS is not integrated LUFS; no loudness standard or perceptual quality is certified.

In [ ]:
def generate_nature_audio(category, duration_seconds, seed=None):
    """Run pretrained inference and return a mono float32 waveform and metadata."""
    if category not in PROMPTS:
        raise ValueError('Unknown category')
    if not 1 <= duration_seconds <= 10:
        raise ValueError('Generate 1 to 10 seconds per call; use the loop workflow for longer audio')
    seed = SEED if seed is None else int(seed)
    result = pipe(prompt=PROMPTS[category], negative_prompt=NEGATIVE,
                  audio_length_in_s=float(duration_seconds), num_inference_steps=INFERENCE_STEPS,
                  guidance_scale=GUIDANCE_SCALE, generator=torch.Generator(device=DEVICE).manual_seed(seed))
    audio = np.asarray(result.audios[0], dtype=np.float32).reshape(-1)
    if not np.isfinite(audio).all() or not len(audio):
        raise ValueError('Invalid model output')
    if MODEL_SR != SR:
        audio = librosa.resample(audio, orig_sr=MODEL_SR, target_sr=SR)
    return audio, {'category': category, 'seed': seed, 'sample_rate': SR,
                   'model': MODEL_ID, 'revision': MODEL_REVISION, 'prompt': PROMPTS[category],
                   'negative_prompt': NEGATIVE, 'steps': INFERENCE_STEPS, 'guidance': GUIDANCE_SCALE}

# Deterministic signal-processing checks; synthetic signals are test fixtures only.
test_t = np.arange(SR * 3) / SR
test_signal = (0.1 * np.sin(2 * np.pi * 220 * test_t)).astype(np.float32)
assert abs(audio_metrics(test_signal)['rms'] - 0.07071) < 0.001
test_loop = loop_crossfade(test_signal, seconds=5, fade_seconds=0.25)
assert len(test_loop) == 5 * SR and np.isfinite(test_loop).all()
assert abs(float(test_loop[0] - test_loop[-1])) < 1e-6
assert np.max(np.abs(soft_master(test_signal))) <= 0.881
print('Signal-processing tests passed')

## 10 · Automated export validation

These assertions check the delivered audio files and manifest. They cannot certify artistic quality, so the preceding listening comparison remains part of acceptance.

In [ ]:
checks=[]
for row in selected:
    path=OUTPUT/row['wav']
    y,sr=sf.read(path,dtype='float32')
    stats=audio_metrics(y,sr)
    ok=(sr==SR and abs(stats['duration_s']-SCENE_SECONDS)<0.02 and
        np.isfinite(y).all() and stats['peak']<=0.92 and stats['rms']>0.001)
    checks.append(dict(category=row['category'],passed=bool(ok),**stats))
report=pd.DataFrame(checks)
display(report)
assert report.passed.all(),'Export validation failed; inspect report before deployment.'
package={'project':'SoundScape AI','version':'1.0','audio_format':'mono PCM16 WAV',
         'sample_rate':SR,'duration_seconds':SCENE_SECONDS,'generator':MODEL_ID,
         'inference_steps':INFERENCE_STEPS, 'guidance_scale':GUIDANCE_SCALE,
         'negative_prompt':NEGATIVE, 'model_revision':MODEL_REVISION, 'model_license':'CC-BY-NC-SA-4.0',
         'selection':'technical screening with optional human override',
         'listening_review':'not recorded',
         'quality_note':'Technical checks passed; listening review is still required.',
         'classes':selected,'demo_mix':'demo_mix_60s.wav'}
with open(OUTPUT/'manifest.json','w') as f:json.dump(package,f,indent=2)
report.to_csv(OUTPUT/'export_validation.csv',index=False)
import zipfile
with zipfile.ZipFile('soundscape_nature_platform_export.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT.glob('*'):
        if path.name.endswith(('_ambient_60s.wav','.json','.csv')) or path.name=='demo_mix_60s.wav':
            z.write(path,arcname=path.name)
print('READY: soundscape_nature_platform_export.zip')

### Exporting the generated files

All WAVs and metadata are stored in `soundscape_export/`. If the notebook creates `soundscape_nature_platform_export.zip`, download it from the Colab Files sidebar or use the optional cell below. Files in the Colab runtime may be lost when the session ends.

In [ ]:
# Optional: run this cell manually in Google Colab to download the export archive.
# from google.colab import files
# files.download('soundscape_nature_platform_export.zip')

## 11 · Deployment contract and limitations

**Output:** `soundscape_nature_platform_export.zip` contains five 60-second mono 16 kHz WAV layers, a demo mix, a manifest, and QA reports. The platform can use the WAV files directly; no model server is required for playback/mixing. To generate new variations on demand, deploy the AudioLDM 2 pipeline with GPU access.

**Evaluation:** visual waveform and spectrogram review, source-recording audit, candidate listening comparisons, technical quality reports, and reproducible generation seeds. Perceptual comfort and prompt adherence must still be checked by listening; no objective metric guarantees a relaxing result.

**Practical limits:** the 60-second exports are crossfaded loops of shorter AI-generated clips. Long-form originality, perfect looping, and studio-grade sound are not guaranteed. The pretrained model download and GPU execution are required for real generation. The ESC-50 audit does not grant commercial reuse rights to its recordings.

## Verification and reproducibility notes

**Observed in the user's Google Colab session (14 November 2026):** Python 3.13.16, CUDA-enabled Tesla T4, PyTorch 2.14.1+cu130, Diffusers 0.42.1, Transformers 5.22.1, Hugging Face Hub 1.33.0, Tokenizers 0.23.3, Librosa 0.11.0, and Accelerate 1.15.0. The GPT2Model → GPT2LMHeadModel compatibility replacement reported successful strict weight loading. Ten 10-second audio candidates were generated (two per class), and five 60-second WAV layers were exported by mastering and looping those candidates. The user listened to the outputs and reported they sounded excellent. These are **pretrained model inference** results, not fine-tuning.

**Still to verify on another runtime:** the pinned installation cell and a complete fresh-session **Run all**. This notebook is prepared for sharing, but cross-environment execution is not guaranteed. The first model download requires internet access and several GB of disk space.

**Important licensing:** ESC-50 is CC BY-NC and the AudioLDM 2 checkpoint is CC BY-NC-SA 4.0; review permissions before commercial use. This notebook does not establish commercial deployment rights.
